in this notebook we compare the results of both fuzzy inference systems side by side to analyze the differences and similarities between the two methods

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

df = pd.read_csv("../data/gtd_processed.csv")

prop_map = {1: 3, 2: 2, 3: 1, 4: 0}
df["prop_inverted"] = df["propextent"].map(prop_map)

print(f"Loaded {len(df):,} rows")

In [ ]:
def trimf(x, a, b, c):
    return np.maximum(0, np.minimum((x - a) / (b - a + 1e-9),
                                     (c - x) / (c - b + 1e-9)))

def trapmf(x, a, b, c, d):
    return np.maximum(0, np.minimum(
        np.minimum((x - a) / (b - a + 1e-9), 1),
        (d - x) / (d - c + 1e-9)
    ))

def fuzzify_nkill(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 0, 0, 1, 4)[0]),
        "Medium":  float(trimf(x, 2, 6, 12)[0]),
        "High":    float(trimf(x, 6, 15, 30)[0]),
        "Extreme": float(trapmf(x, 25, 40, 50, 50)[0]),
    }

def fuzzify_nwound(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 0, 0, 2, 6)[0]),
        "Medium":  float(trimf(x, 3, 10, 20)[0]),
        "High":    float(trimf(x, 15, 35, 60)[0]),
        "Extreme": float(trapmf(x, 45, 65, 80, 80)[0]),
    }

def fuzzify_propextent(val):
    x = np.array([val], dtype=float)
    return {
        "None":         float(trapmf(x, 0, 0, 0, 0.5)[0]),
        "Minor":        float(trimf(x, 0.5, 1, 1.5)[0]),
        "Major":        float(trimf(x, 1.5, 2, 2.5)[0]),
        "Catastrophic": float(trapmf(x, 2.5, 3, 3, 3)[0]),
    }

def fuzzify_attack(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 1, 1, 1, 1.8)[0]),
        "Medium":  float(trimf(x, 1.5, 2, 2.5)[0]),
        "High":    float(trimf(x, 2.5, 3, 3.5)[0]),
        "Extreme": float(trapmf(x, 3.2, 3.6, 5, 5)[0]),
    }

def fuzzify_weapon(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 1, 1, 1, 1.8)[0]),
        "Medium":  float(trimf(x, 1.5, 2, 2.5)[0]),
        "High":    float(trimf(x, 2.5, 3, 3.5)[0]),
        "Extreme": float(trapmf(x, 3.2, 3.6, 5, 5)[0]),
    }

rules = [
    # low severity
    ("Low",    "Low",    "None",         "Low",    "Low",    "Low"),
    ("Low",    "Low",    "None",         "Low",    "Medium", "Low"),
    ("Low",    "Low",    "Minor",        "Low",    "Low",    "Low"),

    # medium severity
    ("Low",    "Low",    "Major",        "Low",    "Low",    "Medium"),
    ("Medium", "Low",    "None",         "Low",    "Low",    "Medium"),
    ("Medium", "Medium", "Minor",        "Medium", "Medium", "Medium"),
    ("Low",    "Medium", "Major",        "Low",    "Medium", "Medium"),
    ("Low",    "Low",    "None",         "High",   "High",   "Medium"),
    ("Low",    "Low",    "Minor",        "Medium", "High",   "Medium"),
    ("Low",    "Low",    "None",         "Medium", "Medium", "Medium"),
    ("Low",    "Low",    "None",         "High",   "Medium", "Medium"),
    ("Low",    "Low",    "None",         "Medium", "High",   "Medium"),
    ("Low",    "Low",    "Minor",        "High",   "Medium", "Medium"),
    ("Low",    "Low",    "None",         "Extreme","Medium", "Medium"),
    ("Low",    "Low",    "None",         "Medium", "Extreme","Medium"),

    # high severity
    ("Medium", "Medium", "Major",        "Medium", "High",   "High"),
    ("High",   "Low",    "Minor",        "High",   "Medium", "High"),
    ("High",   "Medium", "None",         "High",   "High",   "High"),
    ("Medium", "High",   "Major",        "Medium", "High",   "High"),
    ("High",   "High",   "Minor",        "High",   "Medium", "High"),
    ("Low",    "High",   "Major",        "Extreme","High",   "High"),
    ("Medium", "Low",    "Major",        "High",   "Extreme","High"),
    ("Low",    "Low",    "None",         "Extreme","Extreme","High"),
    ("Low",    "Low",    "Minor",        "Extreme","High",   "High"),
    ("Medium", "Low",    "None",         "Extreme","High",   "High"),
    ("Low",    "Medium", "None",         "Extreme","Extreme","High"),

    # critical severity
    ("High",    "Medium",  "Major",        "High",    "Extreme", "Critical"),
    ("High",    "High",    "Major",        "Extreme", "Extreme", "Critical"),
    ("Extreme", "High",    "Major",        "Extreme", "High",    "Critical"),
    ("Extreme", "Extreme", "Catastrophic", "Extreme", "Extreme", "Critical"),
    ("High",    "High",    "Catastrophic", "High",    "Extreme", "Critical"),
    ("Extreme", "Medium",  "Major",        "Extreme", "High",    "Critical"),

    # critical severity added to cover High attack + High weapon combinations
    ("Extreme", "High",    "Major",        "High",    "High",    "Critical"),
    ("Extreme", "Medium",  "Major",        "High",    "High",    "Critical"),
    ("High",    "High",    "Major",        "High",    "High",    "Critical"),
    ("Extreme", "High",    "None",         "High",    "High",    "Critical"),
]

x_out = np.linspace(0, 100, 1000)

output_mf = {
    "Low":      trapmf(x_out, 0, 0, 15, 30),
    "Medium":   trimf(x_out, 20, 40, 55),
    "High":     trimf(x_out, 45, 60, 75),
    "Critical": trapmf(x_out, 65, 80, 100, 100),
}

crisp_output = {
    "Low":      12.5,
    "Medium":   37.5,
    "High":     62.5,
    "Critical": 87.5,
}

def mamdani_infer(nkill_val, nwound_val, prop_val, atk_val, wpn_val):
    fk  = fuzzify_nkill(nkill_val)
    fw  = fuzzify_nwound(nwound_val)
    fp  = fuzzify_propextent(prop_val)
    fa  = fuzzify_attack(atk_val)
    fwp = fuzzify_weapon(wpn_val)
    aggregated = np.zeros_like(x_out)
    for (k, w, p, a, wp, out) in rules:
        strength = min(fk[k], fw[w], fp[p], fa[a], fwp[wp])
        clipped  = np.minimum(strength, output_mf[out])
        aggregated = np.maximum(aggregated, clipped)
    return aggregated

def defuzzify_centroid(aggregated):
    denom = np.sum(aggregated)
    if denom == 0:
        return 0.0
    return float(np.sum(x_out * aggregated) / denom)

def sugeno_infer(nkill_val, nwound_val, prop_val, atk_val, wpn_val):
    fk  = fuzzify_nkill(nkill_val)
    fw  = fuzzify_nwound(nwound_val)
    fp  = fuzzify_propextent(prop_val)
    fa  = fuzzify_attack(atk_val)
    fwp = fuzzify_weapon(wpn_val)
    numerator   = 0.0
    denominator = 0.0
    for (k, w, p, a, wp, out) in rules:
        strength     = min(fk[k], fw[w], fp[p], fa[a], fwp[wp])
        numerator   += strength * crisp_output[out]
        denominator += strength
    if denominator == 0:
        return 0.0
    return numerator / denominator

def score_to_label(score):
    if score < 25:
        return "Low"
    elif score < 50:
        return "Medium"
    elif score < 75:
        return "High"
    else:
        return "Critical"

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()

df["mamdani_score"] = df.progress_apply(
    lambda row: defuzzify_centroid(mamdani_infer(
        row["nkill"], row["nwound"], row["prop_inverted"],
        row["attack_encoded"], row["weapon_encoded"]
    )), axis=1)

df["sugeno_score"] = df.progress_apply(
    lambda row: sugeno_infer(
        row["nkill"], row["nwound"], row["prop_inverted"],
        row["attack_encoded"], row["weapon_encoded"]
    ), axis=1)

df["mamdani_label"] = df["mamdani_score"].apply(score_to_label)
df["sugeno_label"]  = df["sugeno_score"].apply(score_to_label)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["mamdani_score"], bins=50, color="#c0392b", edgecolor="white")
axes[0].set_title("Mamdani Score Distribution")
axes[0].set_xlabel("Severity score")
axes[0].set_ylabel("Count")

axes[1].hist(df["sugeno_score"], bins=50, color="#2980b9", edgecolor="white")
axes[1].set_title("Sugeno Score Distribution")
axes[1].set_xlabel("Severity score")

plt.tight_layout()
plt.show()

In [ ]:
order = ["Low", "Medium", "High", "Critical"]

mamdani_counts = df["mamdani_label"].value_counts().reindex(order)
sugeno_counts  = df["sugeno_label"].value_counts().reindex(order)

x = np.arange(len(order))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, mamdani_counts, width, label="Mamdani", color="#c0392b")
ax.bar(x + width/2, sugeno_counts,  width, label="Sugeno",  color="#2980b9")
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_title("Label Distribution: Mamdani vs Sugeno")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
y_true = df["severity_index"]
order  = ["Low", "Medium", "High", "Critical"]

acc_mamdani = accuracy_score(y_true, df["mamdani_label"])
acc_sugeno  = accuracy_score(y_true, df["sugeno_label"])

print(f"Mamdani Accuracy : {acc_mamdani:.4f} ({acc_mamdani*100:.2f}%)")
print(f"Sugeno Accuracy  : {acc_sugeno:.4f}  ({acc_sugeno*100:.2f}%)")
print()
print("Mamdani Classification Report:")
print(classification_report(y_true, df["mamdani_label"], labels=order, target_names=order))
print("Sugeno Classification Report:")
print(classification_report(y_true, df["sugeno_label"], labels=order, target_names=order))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cm_mamdani = confusion_matrix(y_true, df["mamdani_label"], labels=order)
cm_sugeno  = confusion_matrix(y_true, df["sugeno_label"],  labels=order)

ConfusionMatrixDisplay(cm_mamdani, display_labels=order).plot(ax=axes[0], cmap="Reds",  colorbar=False)
ConfusionMatrixDisplay(cm_sugeno,  display_labels=order).plot(ax=axes[1], cmap="Blues", colorbar=False)

axes[0].set_title("Mamdani Confusion Matrix")
axes[1].set_title("Sugeno Confusion Matrix")

plt.tight_layout()
plt.show()

In [ ]:
df["score_diff"] = df["mamdani_score"] - df["sugeno_score"]

print(f"Mean score difference  : {df['score_diff'].mean():.4f}")
print(f"Max score difference   : {df['score_diff'].max():.4f}")
print(f"Min score difference   : {df['score_diff'].min():.4f}")
print(f"Rows with same label   : {(df['mamdani_label'] == df['sugeno_label']).sum():,}")
print(f"Rows with diff label   : {(df['mamdani_label'] != df['sugeno_label']).sum():,}")